# RSI-Plateau: Kaggle GPU Validation (Real Model Path)

Free alternative to Colab. Kaggle gives ~**30 GPU-hours/week** (T4 x2 or P100),
which is plenty for this smoke test.

### Before you run
1. **Right sidebar -> Settings**
   - **Accelerator: GPU T4 x2** (or P100)
   - **Internet: On** (required to clone the repo and download the model)
2. **Run all** cells.

**Goal:** prove the real pipeline works on a GPU (generation -> oracle filtering ->
LoRA SFT -> eval) before spending budget on Tier-1. These are **plumbing numbers,
not paper numbers** (PRD section 5.9, Tier 0). ~10-20 min.

> Tip: use **Save Version -> Run in background** to run it without keeping the tab open.
> Working outputs persist under `/kaggle/working/`.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Clone the repo into the writable working dir
import os
os.chdir("/kaggle/working")
REPO = "https://github.com/AbhiGuru25/Recursive-self-improvement-RSI.git"
if not os.path.isdir("/kaggle/working/RSI"):
    !git clone -q $REPO /kaggle/working/RSI
%cd /kaggle/working/RSI

In [ ]:
# 3. Install. Kaggle ships torch/transformers/datasets; add the missing training libs only
#    to avoid clobbering the preinstalled CUDA-matched torch.
!pip install -q -e . --no-deps
!pip install -q peft trl accelerate sentence-transformers
# 3b. PEFT hard-errors on Kaggle's old preinstalled torchao (it demands >=0.16).
#     LoRA does not use torchao, so removing it makes PEFT skip that path cleanly.
!pip uninstall -y torchao

In [ ]:
# 4. Sanity: torch sees the GPU and picks a supported dtype
import torch
assert torch.cuda.is_available(), "No GPU. Right sidebar -> Settings -> Accelerator -> GPU."
print("cuda:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("free VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
# 5. Write a GPU smoke config (0.5B, small subset, auto precision)
import pathlib, yaml

cfg = {
    "run": {"name": "kaggle_smoke", "seed": 0, "rounds": 2,
            "output_dir": "artifacts/kaggle_smoke", "use_wandb": False},
    "data": {"dataset": "gsm8k", "train_size": 50, "test_size": 100},
    "model": {"policy": "Qwen/Qwen2.5-0.5B-Instruct", "dtype": "float16",
              "device": "cuda", "judge": None},
    "generation": {"k_samples": 2, "temperature": 0.8, "top_p": 0.95,
                   "max_new_tokens": 256, "batch_size": 8},
    "verifier": {"type": "oracle", "match_acceptance_rate": None, "calibration_size": 0},
    "loop": {"architecture": "star", "frozen_reference_diversity": True},
    "training": {"method": "lora", "lora_r": 8, "lora_alpha": 16, "lora_dropout": 0.05,
                 "learning_rate": 0.0001, "epochs": 1, "batch_size": 2, "grad_accum": 4,
                 "max_seq_len": 512, "precision": "auto"},
    "diversity": {"output_entropy": True, "embedding_clustering": False, "self_bleu": True},
    "stats": {"bootstrap_samples": 200, "ci_alpha": 0.05,
              "plateau_abs_delta": 0.005, "plateau_consecutive": 2},
}
pathlib.Path("configs").mkdir(exist_ok=True)
with open("configs/kaggle_smoke.yaml", "w") as fh:
    yaml.safe_dump(cfg, fh)
print("wrote configs/kaggle_smoke.yaml")

In [ ]:
# 6. Run the real loop (HF generation + LoRA SFT)
!python scripts/run_loop.py --config configs/kaggle_smoke.yaml 2>&1 | tail -40

In [ ]:
# 7. Inspect the result artifact
import json
d = json.load(open("artifacts/kaggle_smoke/result.json"))
print("env:", d["metadata"].get("gpu"))
for r in d["result"]["rounds"]:
    print(r["round"], "acc=", round(r["accuracy"], 3),
          "kept=", f"{r['n_train_kept']}/{r['n_train_total']}",
          "self_bleu=", round(r["diversity"].get("self_bleu", -1), 3),
          "train_loss=", r.get("train_loss"))
print("plateau:", d["result"]["plateau"])

## Success = every stage ran

Generation produced samples, oracle filtering kept some, LoRA training completed,
evaluation produced a number, and `result.json` exists. Low accuracy is expected
(0.5B model, 2 rounds, 50 questions). This de-risks the backends before Tier-1.

### If something fails
- **CUDA OOM:** cell 5 -> `"batch_size": 1`, `"max_new_tokens": 128`.
- **Zero kept samples:** normal for a 0.5B model. Try `Qwen/Qwen2.5-1.5B-Instruct`.
- **pip conflicts:** the install cell uses `--no-deps` for the package to protect
  Kaggle's CUDA-matched torch; only peft/trl/accelerate are added.